In [1]:
def make_greetings(state):
    """
    Read the state and return a greeting message.
    Returns only the feild we want to update. """

    name = state["name"]
    return {"greeting": f"Hello, {name}!"}

current_state = {"name": "Alice","greeting": ""}
update = make_greetings(current_state)
print(update)  # Output: {'greeting': 'Hello, Alice!'}

{'greeting': 'Hello, Alice!'}


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    value: int

def double(state: State):
    """Double the value in the state."""
    return {"value": state["value"] * 2}

def add_ten(state: State):
    """Add ten to the value in the state."""
    return {"value": state["value"] + 10}

#Nodes
builder = StateGraph(State)
builder.add_node("double", double)
builder.add_node("add_ten", add_ten)

#Edges
builder.add_edge(START, "double")
builder.add_edge("double", "add_ten")
builder.add_edge("add_ten", END)

graph = builder.compile()

result = graph.invoke({"value": 5})
print(result)

{'value': 20}


# First Application

In [3]:
import re 
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class TextState(TypedDict):
    raw_text: str
    cleaned: str
    word_count: int
    summary: str

def clean_text(state: TextState) ->dict:
    # Normalize whitespace and remove special characters
    cleaned = re.sub(r'\s+', ' ', state["raw_text"].strip()).lower()
    return {"cleaned": cleaned}

def count_words(state: TextState) -> dict:
    # Count the number of words in the cleaned text
    word_count = len(state["cleaned"].split())
    return {"word_count": word_count}

def summarize_text(state: TextState) -> dict:
    # Create a simple summary by taking the first 10 words
    preview = state['cleaned'][:50]+("..." if len(state['cleaned']) > 50 else '')
    summary =(
        '=== Text Analysis Summary ===\n'
        f"Word Count: {state['word_count']}\n"
        f"Preview: {preview}"
        '=============================='
    )
    return {"summary": summary}

# Build the graph
builder = StateGraph(TextState)
builder.add_node("clean_text", clean_text)
builder.add_node("count_words", count_words)
builder.add_node("summarize_text", summarize_text) 

# Define edges to create the flow of operations
builder.add_edge(START, "clean_text")
builder.add_edge("clean_text", "count_words")
builder.add_edge("count_words", "summarize_text")
builder.add_edge("summarize_text", END)

graph = builder.compile()

#Example
result = graph.invoke({"raw_text": " Hello World! This is a sample text for LangGraph. "})
print(result["summary"])

=== Text Analysis Summary ===
Word Count: 9
Preview: hello world! this is a sample text for langgraph.==============================


# State Management deep dive

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class BadListState(TypedDict):
    steps: list

def step_one(state: BadListState) -> dict:
    
    current = state["steps"]
    new_list = current + ["Step_one"]
    print(f"After Step One: {new_list}")
    return {"steps": new_list}

def step_two(state: BadListState) -> dict:
    current = state["steps"]
    new_list = current + ["Step_two"]
    print(f"After Step Two: {new_list}")
    return {"steps": new_list}

# Build the graph
builder = StateGraph(BadListState)
builder.add_node("step_one", step_one)
builder.add_node("step_two", step_two)  
builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", END)

graph = builder.compile()

result = graph.invoke({"steps": []})
print(f"Final Result: {result['steps']}")

After Step One: ['Step_one']
After Step Two: ['Step_one', 'Step_two']
Final Result: ['Step_one', 'Step_two']


# Reduser

In [5]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END


class GoodListState(TypedDict):
    steps: Annotated[list, add]
    
def step_one(state: GoodListState) -> dict:
    
    current = state["steps"]
    new_list = current + ["Step_one"]
    print(f"After Step One: {new_list}")
    return {"steps": new_list}

def step_two(state: GoodListState) -> dict:
    current = state["steps"]
    new_list = current + ["Step_two"]
    print(f"After Step Two: {new_list}")
    return {"steps": new_list}

# Build the graph
builder = StateGraph(GoodListState)
builder.add_node("step_one", step_one)
builder.add_node("step_two", step_two)  
builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", END)

graph = builder.compile()

result = graph.invoke({"steps": []})
print(f"Final Result: {result['steps']}")

After Step One: ['Step_one']
After Step Two: ['Step_one', 'Step_two']
Final Result: ['Step_one', 'Step_one', 'Step_two']


# Add Messages 

In [7]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    turn: int 

def node_human_input(state: ChatState)-> dict:
    new_msg = HumanMessage(content="What is the capital of France?")
    print(f" [human_input] appending: {new_msg.content!r}")
    return{
        "messages": [new_msg],
        "turn": state["turn"] + 1
    }

def node_ai_reply(state: ChatState)-> dict:
    new_msg = AIMessage(content="The capital of France is Paris.")
    print(f" [ai_reply] appending: {new_msg.content!r}")
    return {"messages": [new_msg]}

builder = StateGraph(ChatState)

builder.add_node("human_input", node_human_input)
builder.add_node("ai_reply", node_ai_reply)

builder.add_edge(START, "human_input")
builder.add_edge("human_input", "ai_reply")
builder.add_edge("ai_reply", END)

graph = builder.compile()

result = graph.invoke({"messages": [], "turn": 0})

print("\nFinal turn count:"), result["turn"]
print("Final messages:")
for msg in result["messages"]:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f" - {role}: {msg.content}")

 [human_input] appending: 'What is the capital of France?'
 [ai_reply] appending: 'The capital of France is Paris.'

Final turn count:
Final messages:
 - Human: What is the capital of France?
 - AI: The capital of France is Paris.
